<a href="https://colab.research.google.com/github/Thashmikab/london-smart-meter-analysis/blob/main/notebooks/04_did_tou_cut_carbon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 · Did time-of-use pricing cut carbon, and would shifting evening demand have helped?

Notebook 03 showed that time-of-use households used a little less electricity in 2013 and paid a little less. This notebook asks whether that cut carbon, and then tests a common assumption: that moving household demand out of the evening peak into the night is good for the climate.

**The answers, in plain terms**

- **The tariff's response didn't measurably cut carbon.** Time-of-use households' emissions fell about 1.1% more than standard households', but that's within noise. Their electricity was also no cleaner per kWh: they didn't move use towards cleaner times.
- **That's because the prices weren't aimed at carbon.** High-price half-hours were no dirtier than normal ones (474.6 against 474.5 g CO₂/kWh), and low-price half-hours, when households were encouraged to use more, were slightly dirtier (479.2). Even a perfect response couldn't have cut carbon through timing.
- **Shifting evening demand overnight wouldn't have helped either.** If every London home had moved 10% of its evening use (16:00–19:00) to 00:00–06:00, about 211 GWh a year would have changed time.
  - Judged by the average grid mix, which barely changed through the day in 2013, it would have saved about **110 tonnes of CO₂ a year**: effectively nothing.
  - Judged by the plants that actually change output when demand moves, it would have **added roughly 60,000–74,000 tonnes a year**, about 18–23 kg per household. Overnight changes in demand were met by plants emitting around 600 g/kWh, consistent with coal at the margin, while evening changes were met by plants at around 320 g/kWh.
- **The lesson: off-peak is not the same as low-carbon.** On the 2013 grid, easing the evening peak, which helps the network, would have worked against carbon. Shifting only cuts emissions when it's aimed at genuinely cleaner hours. Britain's last coal plant closed in 2024, so today's grid behaves differently.

**Inputs:** `household_day` and `generation_mix` from notebook 02.

## Setup

In [ ]:
!pip install pyspark

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from google.colab import drive
from pyspark.sql import SparkSession
from pyspark.sql import functions as F, Window
from pyspark.sql.functions import col, count, when

In [ ]:
#@title Start Spark and load the tables from Drive

spark = SparkSession.builder.appName("london-smart-meters-carbon").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/london-smart-meters"

household_day = spark.read.parquet(f"{DRIVE}/household_day")
genMix_df = spark.read.parquet(f"{DRIVE}/generation_mix")

print(f"household_day: {household_day.count():,} rows")     # 3,490,380
print(f"generation_mix: {genMix_df.count():,} rows")         # 40,848
genMix_df.printSchema()

## 1. Did time-of-use households cut carbon more than standard households?

The same comparison as notebook 03 (same households, July–December 2012 against 2013, each group's change compared), applied to two carbon measures:

- **kg CO₂ per day**, using each half-hour's average grid intensity (calculated in notebook 02).
- **g CO₂ per kWh**, the household's total emissions divided by its total use. It's lower if a household uses more of its electricity at cleaner times, so it isolates *timing* from *amount*.

| Measure | ToU change | Std change | Effect | 95% CI | % of baseline | p |
|---|---:|---:|---:|---|---:|---:|
| kg CO₂ per day | −0.862 | −0.808 | −0.054 | −0.147 to 0.039 | −1.1% | 0.25 |
| g CO₂ per kWh | −47.44 | −46.99 | −0.45 | −1.08 to 0.18 | −0.1% | 0.16 |

Neither effect is significant. The large drop in both groups (about 47 g/kWh, 9%) is the grid itself getting cleaner between the two periods, which affected everyone equally.

In [ ]:
#@title Part 1: did ToU households cut carbon more than standard households?

# Same sample as notebook 03: flagged households, July-December 2012 and 2013, no zero days
sample = (household_day
    .filter(col("in_tariff_comparison") & ~col("zero_day"))
    .filter(col("year").isin(2012, 2013) & (col("month") >= 7))
    .withColumn("period", F.when(col("year") == 2012, "baseline_2012").otherwise("tariff_2013")))

# One row per household per period
hh_period = (sample.groupBy("household_id", "tariff", "period")
    .agg(F.avg("kg_co2").alias("kg_co2"),                                   # average kg CO2 per day
         (F.sum("kg_co2") / F.sum("kwh") * 1000).alias("g_per_kwh")))       # carbon per kWh used

wide = (hh_period.groupBy("household_id", "tariff")
        .pivot("period", ["baseline_2012", "tariff_2013"])
        .agg(F.first("kg_co2").alias("kg_co2"), F.first("g_per_kwh").alias("g_per_kwh")))

pdf = wide.toPandas()
print(pdf.shape)       # 5,208 households, as in notebook 03

metrics = {"kg_co2": "kg CO2 per day", "g_per_kwh": "g CO2 per kWh"}

for m in metrics:
    pdf[f"change_{m}"] = pdf[f"tariff_2013_{m}"] - pdf[f"baseline_2012_{m}"]

print("Group averages")
print(pdf.groupby("tariff")[[f"{p}_{m}" for m in metrics for p in ["baseline_2012", "tariff_2013"]]].mean().round(4).T)

rows = []
for m, label in metrics.items():
    tou = pdf.loc[pdf.tariff == "ToU", f"change_{m}"].dropna()
    std = pdf.loc[pdf.tariff == "Std", f"change_{m}"].dropna()
    effect = tou.mean() - std.mean()
    se = np.sqrt(tou.var() / len(tou) + std.var() / len(std))
    p = stats.ttest_ind(tou, std, equal_var=False).pvalue
    baseline = pdf.loc[pdf.tariff == "ToU", f"baseline_2012_{m}"].mean()
    rows.append({"measure": label,
                 "ToU change": round(tou.mean(), 4), "Std change": round(std.mean(), 4),
                 "effect": round(effect, 4),
                 "95% CI": f"{effect - 1.96*se:.4f} to {effect + 1.96*se:.4f}",
                 "effect as % of ToU baseline": round(100 * effect / baseline, 2),
                 "p-value": round(p, 4)})

pd.DataFrame(rows)

## 2. Were the price signals pointing at clean or dirty hours?

Each 2013 half-hour's price band joined to that half-hour's grid data (both label half-hours by their start).

| Price band | Half-hours | Grid g CO₂/kWh | Wind (MW) | Coal (MW) |
|---|---:|---:|---:|---:|
| High | 788 | 474.6 | 2,714 | 14,790 |
| Normal | 15,072 | 474.5 | 2,381 | 14,268 |
| Low | 1,660 | 479.2 | 2,801 | 14,964 |

The bands are not aligned with carbon: high-price events fell at ordinary-carbon times, and low-price events slightly dirtier ones. The trial's events were designed for network and supply-balancing purposes, so this isn't a flaw in the trial, but it means the tariff couldn't have cut carbon through timing.

In [ ]:
#@title Were the price signals pointing at clean or dirty hours?

tariffs = spark.read.parquet(f"{DRIVE}/tariffs_2013")

(tariffs.join(genMix_df, tariffs.TariffDateTime == genMix_df.DATETIME)
    .groupBy("Tariff")
    .agg(F.count("*").alias("half_hours"),
         F.round(F.avg("CARBON_INTENSITY"), 1).alias("avg_g_per_kwh"),
         F.round(F.avg(col("WIND") + col("WIND_EMB")), 0).alias("avg_wind_MW"),
         F.round(F.avg("COAL"), 0).alias("avg_coal_MW"))
    .orderBy("Tariff")
    .show())

## 3. Scenario: shift 10% of the evening peak overnight

**Assumptions.** Each standard household moves 10% of its 16:00–19:00 use to 00:00–06:00 (when timed appliances such as washing machines can run), every day of 2013. London has 3,266,173 households (2011 Census, the nearest official count; London grew by 2013, so the London totals are slightly conservative). The trial sample was recruited to be representative of Greater London.

**Two ways to count the carbon.**

- **Average intensity:** the carbon of the whole grid mix in each window.
- **Marginal intensity:** the carbon of the plants that actually change output when demand changes. It is estimated from consecutive half-hours: the change in total grid emissions (generation × intensity) is regressed on the change in total generation, separately for each hour of the day. The slope is the emissions of one extra kWh at that hour. This is the standard approach for estimating marginal emissions from national grid data (Hawkes, 2010), and it's the right measure for asking what a *change* in demand does.

**Result.** The average intensity barely varied through the day in 2013 (474 g/kWh in the evening against 471 overnight). The marginal intensity varied a lot: about 300 g/kWh in the early evening, 600–740 overnight.

In [ ]:
#@title Scenario step 1: grid carbon in the evening peak vs overnight (2013)

g = (genMix_df.filter(F.year("DATETIME") == 2013)
     .withColumn("hour", F.hour("DATETIME"))
     .withColumn("day", F.to_date("DATETIME")))

# Average intensity per day in the two windows
daily_ci = g.groupBy("day").agg(
    F.avg(F.when((col("hour") >= 16) & (col("hour") < 19), col("CARBON_INTENSITY"))).alias("ci_peak"),
    F.avg(F.when(col("hour") < 6, col("CARBON_INTENSITY"))).alias("ci_night"))

# Marginal intensity by hour: how much do total emissions change when total generation changes?
gp = g.select("DATETIME", "hour", "GENERATION", "CARBON_INTENSITY").orderBy("DATETIME").toPandas()
gp["emissions_t_per_h"] = gp["GENERATION"] * gp["CARBON_INTENSITY"] / 1000   # MW x g/kWh -> tonnes per hour
gp["dG"] = gp["GENERATION"].diff()
gp["dE"] = gp["emissions_t_per_h"].diff()
gp = gp.dropna()

mef = gp.groupby("hour")[["dG", "dE"]].apply(lambda d: np.polyfit(d["dG"], d["dE"], 1)[0] * 1000)   # g CO2 per kWh
aef = gp.groupby("hour")["CARBON_INTENSITY"].mean()

by_hour = pd.DataFrame({"average_g_per_kwh": aef, "marginal_g_per_kwh": mef}).round(0)
print(by_hour)

mef_peak, mef_night = mef.loc[16:18].mean(), mef.loc[0:5].mean()
aef_peak, aef_night = aef.loc[16:18].mean(), aef.loc[0:5].mean()
print(f"\nAverage:  peak {aef_peak:.0f}, night {aef_night:.0f}, difference {aef_peak - aef_night:.0f} g/kWh")
print(f"Marginal: peak {mef_peak:.0f}, night {mef_night:.0f}, difference {mef_peak - mef_night:.0f} g/kWh")

**How certain are the marginal estimates?** Each hour's slope comes with a 95% confidence interval (about 730 half-hours per hour). Pooling each window gives **322 g/kWh in the evening (316–328)** against **604 g/kWh overnight (597–611)**. The intervals are far apart, so the direction of the result isn't noise. Because hourly and pooled estimates give somewhat different gaps (349 and 282 g/kWh), the London result is reported as a range.

In [ ]:
#@title How certain are the marginal estimates?

def slope_with_ci(d):
    (slope, intercept), cov = np.polyfit(d["dG"], d["dE"], 1, cov=True)
    se = np.sqrt(cov[0, 0])
    return pd.Series({"marginal_g_per_kwh": slope * 1000,
                      "ci_low": (slope - 1.96 * se) * 1000,
                      "ci_high": (slope + 1.96 * se) * 1000,
                      "half_hours": len(d)})

mef_ci = gp.groupby("hour")[["dG", "dE"]].apply(slope_with_ci).round(0)
print(mef_ci)

# The two windows used in the scenario, pooled
for name, hours in [("Evening peak 16-19", range(16, 19)), ("Overnight 00-06", range(0, 6))]:
    print(name, slope_with_ci(gp[gp["hour"].isin(hours)]).round(0).to_dict())

**The households.** Each standard household's 2013 days are joined to that day's evening and overnight intensity. Each household shifts about 65 kWh a year (10% of its evening-peak use); results are scaled to a full year and then to London, for 5%, 10% and 20% shifts.

In [ ]:
#@title Scenario step 2: shift 10% of standard households' evening peak overnight

SHIFT = 0.10
LONDON_HOUSEHOLDS = 3_266_173      # 2011 Census (GLA); 2013 was slightly higher

std13 = (household_day
    .filter((col("tariff") == "Std") & col("in_2013_analysis") & ~col("zero_day") & (col("year") == 2013))
    .join(daily_ci, "day"))

per_hh = (std13.groupBy("household_id").agg(
        F.sum(SHIFT * col("kwh_peak")).alias("shifted_kwh"),
        F.sum(SHIFT * col("kwh_peak") * (col("ci_peak") - col("ci_night")) / 1000).alias("kg_saved_avg"),
        F.count("*").alias("days"))
    .toPandas())

# Scale each household to a full year
per_hh["shifted_kwh_year"] = per_hh["shifted_kwh"] / per_hh["days"] * 365
per_hh["kg_saved_avg_year"] = per_hh["kg_saved_avg"] / per_hh["days"] * 365
per_hh["kg_saved_marg_year"] = per_hh["shifted_kwh_year"] * (mef_peak - mef_night) / 1000

print(f"Standard households: {len(per_hh):,}")
print(f"kWh shifted per household per year: {per_hh['shifted_kwh_year'].mean():.1f}")
print(f"CO2 saved per household per year: average method {per_hh['kg_saved_avg_year'].mean():.2f} kg, "
      f"marginal method {per_hh['kg_saved_marg_year'].mean():.2f} kg")

# Scale to London, for 5%, 10% and 20% shifts
rows = []
for share in [0.05, 0.10, 0.20]:
    k = share / SHIFT
    rows.append({"share of peak shifted": f"{share:.0%}",
                 "London tonnes CO2 (average)": round(per_hh["kg_saved_avg_year"].mean() * k * LONDON_HOUSEHOLDS / 1000),
                 "London tonnes CO2 (marginal)": round(per_hh["kg_saved_marg_year"].mean() * k * LONDON_HOUSEHOLDS / 1000),
                 "London GWh moved": round(per_hh["shifted_kwh_year"].mean() * k * LONDON_HOUSEHOLDS / 1e6, 1)})
pd.DataFrame(rows)

In the table above, positive tonnes are CO₂ saved and negative are CO₂ added. The next cell states the 10% result the other way round (positive = more CO₂), with both marginal versions.

| 10% of evening use moved overnight | London CO₂ per year | Per household |
|---|---:|---:|
| Average grid mix | about 110 tonnes saved | ≈ 0 |
| Marginal plants, pooled windows | about 59,500 tonnes **added** | about 18 kg added |
| Marginal plants, hour by hour | about 73,700 tonnes **added** | about 23 kg added |

In [ ]:
#@title The 10% result, both marginal versions (positive = more CO2)

pooled_peak = slope_with_ci(gp[gp["hour"].isin(range(16, 19))])["marginal_g_per_kwh"]
pooled_night = slope_with_ci(gp[gp["hour"].isin(range(0, 6))])["marginal_g_per_kwh"]

kwh_per_hh = per_hh["shifted_kwh_year"].mean()
london_kwh = kwh_per_hh * LONDON_HOUSEHOLDS

results = [("Average grid mix", -per_hh["kg_saved_avg_year"].mean() * LONDON_HOUSEHOLDS / 1000),
           ("Marginal plants\n(evening and night pooled)", london_kwh * (pooled_night - pooled_peak) / 1e6),
           ("Marginal plants\n(hour by hour)", london_kwh * (mef_night - mef_peak) / 1e6)]

for label, tonnes in results:
    print(f"{label.replace(chr(10), ' '):45s} {tonnes:>10,.0f} t/year   {tonnes / LONDON_HOUSEHOLDS * 1000:6.1f} kg per household")

## 4. The results in one figure

Left: average and marginal carbon intensity by hour of day in 2013, with the two scenario windows shaded. Right: the change in London's CO₂ from moving 10% of every home's evening use overnight.

In [ ]:
#@title Figure: off-peak vs low-carbon in 2013

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker
import numpy as np

def make_figure_04(hours, avg_g, marg_g, marg_lo, marg_hi, results, path):
    """hours: 0-23; results: list of (label, tonnes_change) where positive = more CO2."""
    blue, red, grey = "#2f6db3", "#c0392b", "#8a8a8a"
    fig, (a, b) = plt.subplots(1, 2, figsize=(13, 5.2), gridspec_kw={"width_ratios": [1.25, 1]})

    a.axvspan(-0.5, 5.5, color="#f1e3c8", alpha=0.6, lw=0)
    a.axvspan(15.5, 18.5, color="#d7e6f5", alpha=0.8, lw=0)
    a.text(2.5, 222, "Overnight\n00:00–06:00", ha="center", va="bottom", fontsize=9, color="#7a5a1e")
    a.text(17, 222, "Evening peak\n16:00–19:00", ha="center", va="bottom", fontsize=9, color="#1f4e7a")
    a.fill_between(hours, marg_lo, marg_hi, color=red, alpha=0.18, lw=0)
    a.plot(hours, marg_g, "-o", color=red, ms=4, lw=2, label="Marginal: plants that change output\nwhen demand changes")
    a.plot(hours, avg_g, "-o", color=grey, ms=4, lw=2, label="Average: the whole grid mix")
    a.set_xticks(range(0, 24, 3), [f"{h:02d}:00" for h in range(0, 24, 3)])
    a.set_xlim(-0.5, 23.5); a.set_ylim(215, 780)
    a.set_ylabel("Carbon intensity of electricity (g CO₂ per kWh)")
    a.set_xlabel("Time of day (2013)")
    a.set_title("Off-peak wasn't low-carbon in 2013: changes in overnight\ndemand were met by coal, evening changes by cleaner plants",
                loc="left", fontsize=11.5)
    a.legend(frameon=False, fontsize=8.5, loc="upper center", bbox_to_anchor=(0.55, 0.98))

    labels = [r[0] for r in results]; vals = [r[1] for r in results]
    ys = np.arange(len(vals))[::-1]
    colors = [red if v > 0 else blue for v in vals]
    b.barh(ys, vals, color=colors, height=0.55)
    b.axvline(0, color="#444", lw=0.8)
    for y, v in zip(ys, vals):
        txt = f"+{round(v, -2):,.0f} t" if v > 0 else f"≈ 0  ({-v:,.0f} t saved)"
        b.text(max(v, 0) + 1500, y, txt, va="center", ha="left", fontsize=10)
    b.set_yticks(ys, labels, fontsize=9.5)
    b.set_xlim(-12000, max(vals) * 1.3)
    b.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    b.set_xlabel("Change in London's CO₂ per year (tonnes; positive = more CO₂)")
    b.set_title("Shifting 10% of every London home's evening use\novernight would have added CO₂, not saved it",
                loc="left", fontsize=11.5)
    for ax in (a, b):
        for s in ("top", "right"):
            ax.spines[s].set_visible(False)
    b.spines["left"].set_visible(False); b.tick_params(axis="y", length=0)
    fig.text(0.01, 0.01, "Grid data: NESO historic GB generation mix, 2013. Shaded band: 95% confidence interval. "
             "Scenario: 4,230 standard-tariff households scaled to 3.27 million London households (2011 Census).",
             fontsize=8, color="#666")
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    fig.savefig(path, dpi=160)
    return fig


!mkdir -p figures
make_figure_04(list(range(24)),
               by_hour["average_g_per_kwh"].values, mef_ci["marginal_g_per_kwh"].values,
               mef_ci["ci_low"].values, mef_ci["ci_high"].values,
               results, "figures/carbon_off_peak_2013.png")
plt.show()

## What this means

- **The 2013 tariff didn't cut carbon**, and couldn't have through timing, because its price events weren't aligned with the grid's carbon.
- **"Off-peak" was not "low-carbon" in 2013.** The average grid mix was almost the same at every hour, so moving demand around barely changes emissions on that measure. At the margin, overnight demand was met by much dirtier plants than evening demand, so a 10% shift would have added roughly 60,000–74,000 tonnes of CO₂ a year across London.
- **Network goals and carbon goals can pull apart.** Easing the evening peak helps local networks, but on a coal-heavy grid it would have raised emissions. A tariff meant to cut carbon has to target the hours that are actually clean.

**Limits.** The marginal estimates come from half-hour-to-half-hour changes, so they describe how the grid responded to short-term swings in demand; a large, sustained shift could change which plants are scheduled. NESO's intensity figures include its own assumptions for imports and embedded generation. The London figure assumes the standard households represent London homes and uses the 2011 household count.

**Next:** notebook 05 asks what the trial's measured response could have achieved if the price events had been timed by carbon: first with perfect knowledge of the grid, then using a forecast made a day ahead.